# TP7 – $\mathbb Z/n\mathbb Z$, groupe des inversibles et logarithme discret

<font color="red">**Consigne :** lire le sujet dans l'ordre et **exécuter chaque cellule de code pré-remplie** pour observer le résultat avant de continuer.</font>

Dans les TP précédents, on a programmé nous-mêmes les calculs modulo $n$ (inverses, puissances, théorème chinois). SageMath sait aussi manipuler directement l'anneau $\mathbb Z/n\mathbb Z$ et son groupe des inversibles. On s'en sert ici pour étudier la structure de $(\mathbb Z/n\mathbb Z)^\times$, puis pour aborder un nouveau problème difficile, sur lequel repose une partie de la cryptographie : le **logarithme discret**.

**Conventions et fonctions Sage utiles.**

* Pour un élément `g` de $\mathbb Z/n\mathbb Z$ : `g.is_unit()`, `g.multiplicative_order()`, `g^k` (calculé par exponentiation rapide), `~g` (inverse).
* `factor(n)`, `euler_phi(n)`, `random_prime(2**b, proof=False, lbound=2**(b-1))`, `is_prime(n)`, `isqrt(n)` (partie entière de $\sqrt n$), `cputime()`.
* Un *dictionnaire* associe des valeurs à des clés : `D = {}` crée un dictionnaire vide, `D[x] = q` associe la valeur `q` à la clé `x`, `x in D` teste si `x` est une clé de `D`, et `D[x]` renvoie la valeur associée. Le test `x in D` se fait en temps quasi constant, quelle que soit la taille de `D`.
* Les fonctions `h.log(g)`, `discrete_log(h, g)` et `Zmod(p).multiplicative_generator()` de Sage font ce que l'on demande de programmer dans la partie 3 : on ne les utilise que pour **vérifier** nos fonctions et comparer les temps de calcul.

Les cellules *Réponse :* sont à compléter par un texte rédigé.

## 1. L'anneau $\mathbb Z/n\mathbb Z$ dans SageMath

Dans SageMath, on construit l'anneau $\mathbb Z/n\mathbb Z$ avec la commande `IntegerModRing(n)`. Pour calculer modulo $n$, on convertit d'abord les entiers en éléments de cet anneau, en écrivant `A(a)` : les opérations se font ensuite automatiquement modulo $n$.

*Remarque.* Le constructeur `IntegerModRing` possède deux alias (c'est-à-dire deux raccourcis), `Integers` et `Zmod`.

In [ ]:
A = IntegerModRing(6) # ℤ/6ℤ
print(A)
a, b, c, d = A(0), A(1), A(3), A(5)
a+b, b+d, c+d

In [ ]:
IntegerModRing(17) is Integers(17) is Zmod(17) # alias

Chaque élément possède des méthodes qui lui sont associées. On peut par exemple calculer l'ordre d'un élément $g$ dans le groupe additif $(\mathbb Z/n\mathbb Z, +)$ avec `g.additive_order()`, tester s'il est inversible (on dit que c'est une *unité*) avec `g.is_unit()`, et dans ce cas calculer son ordre dans le groupe multiplicatif $\left((\mathbb Z/n\mathbb Z)^\times, \times\right)$ avec `g.multiplicative_order()`. On rappelle que l'*ordre multiplicatif* de $g$ est le plus petit entier $k \ge 1$ tel que $g^k = 1$.

In [ ]:
c.additive_order(), c.is_unit()

In [ ]:
d.additive_order(), d.is_unit(), d.multiplicative_order()

Pour calculer l'inverse d'un élément inversible $g$, plusieurs écritures sont possibles : `1/g`, `g^-1`, `~g` (le symbole `~` signifie « inverse » dans SageMath), ou la méthode `g.inverse_of_unit()`, dont le nom rappelle que l'on ne peut inverser que des unités.

In [ ]:
1/d, d^-1, ~d, d.inverse_of_unit()

In [ ]:
d * ~d == ~d * d == A.one() # one() permet de renvoyer le "1" de l'anneau

On peut aussi obtenir des propriétés de l'anneau lui-même.

In [ ]:
print("A est un anneau :",A.is_ring())
print("Son ordre est :", A.order())
print("A est intègre :", A.is_integral_domain())
print("A est un corps :", A.is_field())

### Exercice 1 — Inversibles et ordres multiplicatifs

1. Dans l'anneau $\mathbb Z/16\mathbb Z$ :
   1. parmi les éléments $2$, $4$, $5$, $6$, $13$ et $15$, lesquels sont inversibles ?
   2. calculer l'ordre multiplicatif et l'inverse de ceux qui sont inversibles.
2. Répondre aux mêmes questions dans l'anneau $\mathbb Z/17\mathbb Z$.
3. Écrire une fonction `ordres(n)` qui renvoie la liste des couples $(g, k)$, où $g$ parcourt les éléments inversibles de $\mathbb Z/n\mathbb Z$ et $k$ est l'ordre multiplicatif de $g$. L'appliquer pour $n = 16$, $17$ et $24$.
4. Comparer les ordres obtenus avec $\varphi(n)$. Que remarque-t-on ? Quel lien avec le théorème d'Euler ? Et avec le théorème de Lagrange ?
5. Dans lesquels de ces trois anneaux existe-t-il un élément d'ordre $\varphi(n)$ ? Que remarque-t-on de surprenant dans $\mathbb Z/24\mathbb Z$ ?

*Réponse :*

## 2. Le groupe des inversibles $(\mathbb Z/n\mathbb Z)^\times$

SageMath sait calculer le groupe des inversibles de $\mathbb Z/n\mathbb Z$ de plusieurs manières. La première consiste à lister ses éléments, à tester s'il est cyclique, ou à calculer son ordre. *Selon les cas, SageMath parle de « multiplicative group » ou de « unit group » : les deux expressions désignent le groupe des inversibles.*

In [ ]:
A = Zmod(36)
A.list_of_elements_of_multiplicative_group(), A.multiplicative_group_is_cyclic(), A.unit_group_order()

L'autre façon est de calculer *vraiment* le groupe des inversibles, avec `A.unit_group()`. On obtient une description un peu cryptique, qui signifie ici : « le groupe des inversibles de `A` est un groupe multiplicatif abélien, isomorphe au groupe $C_2 \times C_6$ », où $C_k$ est un groupe "abstrait". 

Plus précisément, $C_k$ désigne le groupe multiplicatif cyclique d'ordre $k$, qui est isomorphe à $(\mathbb Z/k\mathbb Z, +)$. Le groupe $C_k$ possède un générateur $g$ tel que $g^k = 1$ et $g^\ell \neq 1$ pour $0 < \ell < k$ ; autrement dit, $C_k = \{g^0, g^1, \dots, g^{k-1}\}$.

In [ ]:
G = A.unit_group()
G

Si l'on liste les éléments de $G$, on s'aperçoit qu'ils n'ont rien à voir avec notre $\mathbb Z/36\mathbb Z$ de départ ! SageMath décrit $G$ à l'aide de deux *générateurs*, qu'il a appelés `f0` et `f1`. Le *produit cartésien* $C_2 \times C_6$ s'obtient en prenant tous les produits possibles entre les éléments de $C_2 = <f_0> =  \{1, f_0\}$ et ceux de $C_6 = <f_1> = \{1, f_1, \dots, f_1^5\}$.

In [ ]:
print(G.list())   # on vérifie qu'il y a bien 12 éléments, comme l'indique A.unit_group_order()
f0, f1 = G.gens() # on récupère les générateurs de G
f0^2, f1^6

Il est malgré tout possible de revenir à notre $\mathbb Z/36\mathbb Z$ de départ : chaque élément du groupe $G$ possède une *valeur* dans $\mathbb Z/36\mathbb Z$, c'est-à-dire que chaque élément de $G$ représente un élément de $\mathbb Z/36\mathbb Z$.

In [ ]:
for g in G:
    print(g, "→", g.value())

On vérifie que c'est cohérent : puisque $f_0$ correspond à $19$ et $f_1^3$ à $17$, le produit $f_0 f_1^3$ doit correspondre à $(19 \times 17) \bmod 36 = 35$ ; de même, $(f_0 f_1^3)^2 = f_0^2 f_1^6$ doit valoir $1$.

In [ ]:
f0.value() * (f1^3).value() == A(35), (f0*f1^3).value()^2 == A.one()

### Exercice 2 — Le groupe des inversibles et le théorème chinois

1. Calculer le groupe des inversibles de $\mathbb Z/105\mathbb Z$. Combien de générateurs SageMath utilise-t-il pour le décrire ? À quel élément de $\mathbb Z/105\mathbb Z$ correspond chacun d'eux ?
2. Calculer les groupes des inversibles de $\mathbb Z/3\mathbb Z$, $\mathbb Z/5\mathbb Z$ et $\mathbb Z/7\mathbb Z$, sachant que $105 = 3 \times 5 \times 7$. Que remarque-t-on ? L'expliquer à l'aide du théorème chinois.


*Réponse :*

### Exercice 3 — Quand $(\mathbb Z/n\mathbb Z)^\times$ est-il cyclique ?

1. Calculer le groupe des inversibles de $\mathbb Z/n\mathbb Z$ pour $n = 2, 3, \dots, 30$. Lesquels sont cycliques ? Conjecturer une caractérisation des entiers $n$ pour lesquels $(\mathbb Z/n\mathbb Z)^\times$ est cyclique.
2. Vérifier expérimentalement que si $p > 2$ est premier, $(\mathbb Z/p^e\mathbb Z)^\times$ est cyclique. *Utiliser plusieurs valeurs de $p$ et de $e$, en dépassant $30$.*
3. Ce résultat n'est pas vrai pour $p = 2$. Essayer diverses valeurs de $e$ pour conjecturer la structure de $(\mathbb Z/2^e\mathbb Z)^\times$ en fonction de $e$.

*Réponse :*

## 3. Logarithme discret

Soit $p$ un nombre premier. Étant donné un générateur $g$ de $(\mathbb Z/p\mathbb Z)^\times$ et un élément $h \in (\mathbb Z/p\mathbb Z)^\times$, le *logarithme discret de $h$ en base $g$* est l'unique entier $n \in \{0, \dots, p-2\}$ tel que $g^n = h$.

Calculer $h = g^n$ à partir de $n$ est facile : c'est une exponentiation rapide. Le problème inverse, retrouver $n$ à partir de $h$, est en revanche **difficile** lorsque $p$ est grand et bien choisi. C'est sur cette difficulté que repose l'échange de clés de Diffie–Hellman (1976).

### Exercice 4 — Tirer un nombre premier et un générateur

Écrire une fonction `randgen(b)` qui renvoie un couple $(p, g)$, où $p$ est un nombre premier aléatoire de $b$ bits et $g$ un générateur de $(\mathbb Z/p\mathbb Z)^\times$.

*Réponse :*

### Exercice 5 — L'échange de clés de Diffie–Hellman

Alice et Bob veulent se mettre d'accord sur une clé secrète commune, en ne communiquant que par un canal public.
- Ils choisissent publiquement un nombre premier $p$ et un générateur $g$ de $(\mathbb Z/p\mathbb Z)^\times$.
- Alice tire au hasard un entier secret $a \in \{1, \dots, p-2\}$ et envoie $A = g^a$ à Bob.
- Bob tire au hasard un entier secret $b \in \{1, \dots, p-2\}$ et envoie $B = g^b$ à Alice.
- Alice calcule $B^a$, et Bob calcule $A^b$.

Ce protocole est appelé échange de clé de Diffie-Hellman, du nom de ses créateurs.

1. Montrer qu'Alice et Bob obtiennent le même élément, qui constitue leur clé commune.
2. Simuler l'échange avec un nombre premier de $256$ bits, en utilisant la fonction de l'exercice précédent et des nombres aléatoires ``a`` et ``b``.
3. Une espionne observe $p$, $g$, $A$ et $B$. Expliquer comment elle peut retrouver la clé commune si elle sait calculer des logarithmes discrets.

*Réponse :*

### Exercice 6 — Logarithme discret naïf

1. Écrire une fonction `log_naif(h, g, p)` qui prend en entrée un élément $h \in (\mathbb Z/p\mathbb Z)^\times$ et un générateur $g$ de $(\mathbb Z/p\mathbb Z)^\times$, et qui calcule l'unique entier $n \in \{0, \dots, p-2\}$ tel que $g^n = h$, en calculant successivement $g^0, g^1, g^2, \dots$
2. Quel est le coût de cette fonction en fonction de $p$, dans le pire des cas ?

*Réponse :*

### Exercice 7 — Pas de bébé, pas de géant

On s'intéresse à la stratégie « pas de bébé, pas de géant » (*baby-step giant-step*), due à Shanks. On pose $m = \lfloor \sqrt{p-1} \rfloor$. Si $h = g^n$, on écrit la division euclidienne $n = qm + r$, avec $0 \le r < m$ et $0 \le q \le (p-1)/m$. On a alors $h = g^{qm} g^r$, autrement dit
$$g^{qm} = h \cdot g^{-r}.$$
* **Pas de géant.** On précalcule tous les éléments $g^{qm}$ pour $0 \le q \le (p-1)/m$, et on les range dans un dictionnaire avec la valeur $q$ associée.
* **Pas de bébé.** On calcule ensuite les éléments $h \cdot g^{-r}$ pour $r = 0, 1, 2, \dots$, jusqu'à en trouver un qui figure dans le dictionnaire, c'est-à-dire jusqu'à obtenir une égalité $h \cdot g^{-r} = g^{qm}$. On en déduit $n = qm + r$.

1. Écrire une fonction `pas_de_geants(g, p)` qui renvoie un dictionnaire $G$ tel que $G[g^{qm}] = q$ pour $0 \le q \le (p-1)/m$.
2. Écrire une fonction `log_discret(h, g, p)` qui calcule le logarithme discret de $h$ en base $g$ avec cet algorithme. La vérifier en la comparant à `log_naif` et à `h.log(g)`.
3. Combien, au maximum, doit-on tester de valeurs de $r$ pour être sûr de trouver une égalité ? 
4. Quel est le coût de cet algorithme, en temps et en mémoire, en fonction de $p$ ? Expliquer la différence observée avec `log_naif`.
5. Jusqu'à quelle taille de $p$ peut-on calculer un logarithme discret en moins de $10$ secondes ? Comparer avec `log_naif` et avec la fonction `h.log(g)` de Sage.

*Réponse :*